In [7]:
# Bước 1: Xoá folder cũ nếu có
!rm -rf /kaggle/working/Video_Action_Recognition

# Bước 2: Clone (để yên, đừng bấm Ctrl+C, chờ nó chạy xong)
!git -c credential.helper='' clone --depth 1 \
    https://github.com/buidong941-ship-it/Human-Action-Recognition.git \
    /kaggle/working/Video_Action_Recognition

Cloning into '/kaggle/working/Video_Action_Recognition'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 21 (delta 0), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 85.40 MiB | 18.41 MiB/s, done.


In [8]:
# Cài đặt các thư viện cần thiết
!pip install gradio decord torchvision

In [9]:
import sys
import os
import torch
import torchvision.transforms as transforms
import gradio as gr
import numpy as np
import cv2  
import urllib.request

# ---------------------------------------------------------
# 1. IMPORT KIẾN TRÚC MODEL
# ---------------------------------------------------------
sys.path.append('/kaggle/working/Video_Action_Recognition')

from models.tsm import TSM_Network

# ---------------------------------------------------------
# 2. KHỞI TẠO ODELS & LOAD TRỌNG SỐ (.pt)
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

num_classes = 51   
num_segments = 16  

# Khởi tạo model
model_tsm = TSM_Network(num_classes=num_classes, n_segment=num_segments).to(device)

# Load trọng số
path_tsm = "/kaggle/input/models/midzid/tsm-resnet50-best/pytorch/default/1/tsm_resnet50_best.pt"

model_tsm.load_state_dict(torch.load(path_tsm, map_location=device))

model_tsm.eval()

# ---------------------------------------------------------
# 3. TỪ ĐIỂN NHÃN & TIỀN XỬ LÝ (PRE-PROCESSING)
# ---------------------------------------------------------
LABELS = [
    "brush_hair", "cartwheel", "catch", "chew", "clap", 
    "climb", "climb_stairs", "dive", "draw_sword", "dribble", 
    "drink", "eat", "fall_floor", "fencing", "flic_flac", 
    "golf", "handstand", "hit", "hug", "jump", 
    "kick", "kick_ball", "kiss", "laugh", "pick", 
    "pour", "pullup", "punch", "push", "pushup", 
    "ride_bike", "ride_horse", "run", "shake_hands", "shoot_ball", 
    "shoot_bow", "shoot_gun", "sit", "situp", "smile", 
    "smoke", "somersault", "stand", "swing_baseball", "sword", 
    "sword_exercise", "talk", "throw", "turn", "walk", 
    "wave"
]

# A. Tiền xử lý Ảnh RGB cho TSM
transform = transforms.Compose([
    transforms.ToPILImage(),             
    transforms.Resize((224, 224)),       
    transforms.ToTensor(),               
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Đang chạy trên thiết bị: cpu
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 387MB/s]


In [10]:
def process_video_rgb(video_path, num_frames=16): 
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        return torch.zeros(1, num_frames, 3, 224, 224)
        
    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames, dtype=int)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(idx - 1, 0))
            ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(transform(frame_rgb))
        else:
            frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
    cap.release()
    
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
    
    video_tensor = torch.stack(frames[:num_frames]).unsqueeze(0)  # (1, T, C, H, W)
    return video_tensor
    


In [11]:
# =========================================================
# HÀM TRÍCH XUẤT 16 FRAMES ĐỂ HIỂN THỊ UI
# =========================================================
def extract_frames_for_display(video_path, num_frames=16):
    if not video_path: 
        return []
        
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        return []

    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames, dtype=int)
    frames_for_ui = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(idx - 1, 0))
            ret, frame = cap.read()
            
        if ret:
            # Chuyển BGR (OpenCV) sang RGB để Gradio hiển thị đúng màu
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames_for_ui.append(frame_rgb)
            
    cap.release()
    
    # Padding nếu video quá ngắn không đủ lấy
    while len(frames_for_ui) < num_frames and len(frames_for_ui) > 0:
        frames_for_ui.append(frames_for_ui[-1])
        
    return frames_for_ui[:num_frames]

In [12]:
# ---------------------------------------------------------
# 4. CÁC HÀM DỰ ĐOÁN ĐỘC LẬP
# ---------------------------------------------------------
def get_prediction_text(outputs):
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    predicted_idx = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_idx].item()
    return f"Hành động: **{LABELS[predicted_idx]}**\nĐộ chính xác: {confidence*100:.2f}%"

def predict_tsm(video_filepath):
    if not video_filepath: return "Vui lòng tải video."
    try:
        input_tensor = process_video_rgb(video_filepath).to(device)
        with torch.no_grad():
            return get_prediction_text(model_tsm(input_tensor))
    except Exception as e: return f"Lỗi: {str(e)}"


In [13]:
# ---------------------------------------------------------
# 5. XÂY DỰNG GIAO DIỆN GRADIO BLOCKS
# ---------------------------------------------------------
gr.close_all() 

with gr.Blocks() as demo:
    gr.Markdown("<center><h1>🚀 Hệ Thống Nhận Diện Hành Động</h1></center>")
    gr.Markdown("Tải video lên và trải nghiệm sức mạnh của CNN (TSM).")
    
    # --- TAB 1: TSM ---
    with gr.Tab("🎞️ TSM (ResNet50)"):
        with gr.Row():
            with gr.Column():
                vid_tsm = gr.Video(label="Upload Video Test")
                # THÊM NÚT PREPROCESS
                btn_prep_tsm = gr.Button("🔍 Preprocess (Xem 16 Frames)", variant="secondary")
                btn_tsm = gr.Button("Dự đoán bằng TSM", variant="primary")
            with gr.Column():
                out_tsm = gr.Markdown(label="Kết quả")
                # THÊM GALLERY ĐỂ HIỂN THỊ FRAMES (hiển thị dạng lưới 4x4)
                gal_tsm = gr.Gallery(label="16 Frames đầu vào", columns=4, rows=4, height="auto")
                
        btn_prep_tsm.click(fn=extract_frames_for_display, inputs=vid_tsm, outputs=gal_tsm)
        btn_tsm.click(fn=predict_tsm, inputs=vid_tsm, outputs=out_tsm)

In [14]:
# ---------------------------------------------------------
# 6. KHỞI CHẠY APP
# ---------------------------------------------------------
demo.launch(server_name="0.0.0.0", share=True)

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://0db4695321850d052d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
